# Notebook 2 — Inter-Annotator Agreement (κ) + Gemini Zero-Shot Baseline

Run this **after** two annotators have filled `annotator_1_label` and `annotator_2_label` in
`annotation_sample_BLANK.xlsx`.

It does two independent things and writes both into `outputs/paper_q1/tables/`:

**Part A — Inter-annotator agreement (no API, no GPU, seconds).**
- Cohen's κ between the two annotators.
- Raw percent agreement, per-class agreement, and each annotator vs. the consolidated gold label.
- Writes `table_iaa_kappa.tex` + `.csv`.

**Part B — Zero-shot LLM baseline via OpenRouter (API, no GPU).**
- Classifies a test sample zero-shot into the five classes and scores Macro-F1 / Weighted-F1 /
  Accuracy against gold.
- Uses your `notebooks/.env` (`OPENROUTER_API_KEY`, `OPENROUTER_MODEL`, `OPENROUTER_BASE_URL`) —
  loaded automatically, no keys pasted into the notebook.
- Writes `table_llm_baseline.tex` + `.csv`.
- Cost/time controls are in the config cell (subset size, rate limit).

Run Part A first; it is free and is the higher-priority result. Part B is optional and billed per call.


In [1]:
import sys
!{sys.executable} -m pip install openai python-dotenv


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# PART A — Inter-annotator agreement (Cohen's kappa). No API, no GPU.
# ══════════════════════════════════════════════════════════════════════════════
import os
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

ROOT     = os.environ.get("PROJECT_ROOT", "..")
ANN_DIR  = os.environ.get("KAPPA_DIR", f"{ROOT}/outputs/annotation")
TAB      = f"{ROOT}/outputs/paper_q1/tables"
os.makedirs(TAB, exist_ok=True)
LABELS   = ["none", "abusive", "sexual", "religious", "threat"]

ann_path = os.environ.get("ANNOTATION_XLSX", f"{ANN_DIR}/annotation_sample_BLANK.xlsx")
ann  = pd.read_excel(ann_path, sheet_name="annotate")
gold = pd.read_csv(f"{ANN_DIR}/annotation_sample_GOLD.csv", encoding="utf-8-sig")
print(f"reading annotations from: {ann_path}")

def norm(s):
    return str(s).strip().lower()

for c in ["annotator_1_label", "annotator_2_label"]:
    ann[c] = ann[c].map(norm)

# keep only rows both annotators completed with valid labels
valid = ann["annotator_1_label"].isin(LABELS) & ann["annotator_2_label"].isin(LABELS)
n_missing = int((~valid).sum())
work = ann[valid].merge(gold[["sample_uid", "gold_label"]], on="sample_uid", how="left")
work["gold_label"] = work["gold_label"].map(norm)
print(f"annotated rows used: {len(work)}  (skipped {n_missing} blank/invalid)")
assert len(work) >= 100, "need at least ~100 completed rows for a meaningful kappa"

a1, a2, g = work["annotator_1_label"], work["annotator_2_label"], work["gold_label"]

# Cohen's kappa (annotator 1 vs 2) and each annotator vs gold
kappa_12    = cohen_kappa_score(a1, a2, labels=LABELS)
kappa_1gold = cohen_kappa_score(a1, g,  labels=LABELS)
kappa_2gold = cohen_kappa_score(a2, g,  labels=LABELS)
pct_agree   = float((a1.values == a2.values).mean())

def interpret(k):
    if k < 0.20: return "slight"
    if k < 0.40: return "fair"
    if k < 0.60: return "moderate"
    if k < 0.80: return "substantial"
    return "almost perfect"

print(f"\nCohen's kappa (annotator 1 vs 2): {kappa_12:.3f}  ({interpret(kappa_12)} agreement)")
print(f"percent agreement (1 vs 2)      : {pct_agree*100:.1f}%")
print(f"kappa annotator 1 vs gold       : {kappa_1gold:.3f}")
print(f"kappa annotator 2 vs gold       : {kappa_2gold:.3f}")

# per-class agreement between the two annotators (how often they both pick that class together)
rows = []
for lab in LABELS:
    both = int(((a1 == lab) & (a2 == lab)).sum())
    either = int(((a1 == lab) | (a2 == lab)).sum())
    jacc = both / either if either else float("nan")   # class-wise Jaccard-style agreement
    rows.append({"Class": lab, "Both agree": both, "Either used": either,
                 "Agreement": round(jacc, 3) if either else "—"})
per_class = pd.DataFrame(rows)
print("\nper-class agreement:\n", per_class.to_string(index=False))

# ---- summary table for the paper ----
summary = pd.DataFrame([
    {"Measure": "Cohen's $\\kappa$ (annotator 1 vs.\\ 2)", "Value": f"{kappa_12:.3f}", "Interpretation": interpret(kappa_12)},
    {"Measure": "Percent agreement (1 vs.\\ 2)",            "Value": f"{pct_agree*100:.1f}\\%", "Interpretation": "—"},
    {"Measure": "$\\kappa$ (annotator 1 vs.\\ consolidated label)", "Value": f"{kappa_1gold:.3f}", "Interpretation": interpret(kappa_1gold)},
    {"Measure": "$\\kappa$ (annotator 2 vs.\\ consolidated label)", "Value": f"{kappa_2gold:.3f}", "Interpretation": interpret(kappa_2gold)},
    {"Measure": "Annotated sample size",                   "Value": f"{len(work)}", "Interpretation": "stratified"},
])
summary.to_csv(f"{TAB}/table_iaa_kappa.csv", index=False)

def write_tex(df, path, caption, label, colfmt):
    cols = list(df.columns)
    L = [r"\begin{table}[!htbp]", r"\centering", f"\\caption{{{caption}}}", f"\\label{{{label}}}",
         r"\small", f"\\begin{{tabular}}{{@{{}}{colfmt}@{{}}}}", r"\toprule",
         " & ".join(f"\\textbf{{{c}}}" for c in cols) + r" \\", r"\midrule"]
    for _, r in df.iterrows():
        L.append(" & ".join(str(r[c]) for c in cols) + r" \\")
    L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    open(path, "w").write("\n".join(L))

write_tex(summary, f"{TAB}/table_iaa_kappa.tex",
          "Inter-annotator agreement on a stratified sample of "
          f"{len(work)} comments, two annotators labelling into the five classes independently.",
          "tab:iaa", "lll")
print(f"\n✅ wrote {TAB}/table_iaa_kappa.(csv|tex)")
print(f"\n>>> REPORT THIS IN THE PAPER: Cohen's kappa = {kappa_12:.3f} ({interpret(kappa_12)}), n = {len(work)} <<<")


reading annotations from: ../outputs/annotation/annotation_sample_BLANK.xlsx
annotated rows used: 500  (skipped 0 blank/invalid)

Cohen's kappa (annotator 1 vs 2): 0.709  (substantial agreement)
percent agreement (1 vs 2)      : 80.0%
kappa annotator 1 vs gold       : 0.851
kappa annotator 2 vs gold       : 0.852

per-class agreement:
     Class  Both agree  Either used  Agreement
     none         204          263      0.776
  abusive         103          152      0.678
   sexual          44           84      0.524
religious          38           62      0.613
   threat          11           39      0.282

✅ wrote ../outputs/paper_q1/tables/table_iaa_kappa.(csv|tex)

>>> REPORT THIS IN THE PAPER: Cohen's kappa = 0.709 (substantial), n = 500 <<<


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# PART B — Zero-shot LLM baseline via OpenRouter. API (no GPU).
# ══════════════════════════════════════════════════════════════════════════════
# COST / TIME CONTROLS -- read before running.
#   * N_LLM = 2000 stratified test rows keeps spend well under $1-2 and runs in ~15-25 min.
#   * Set N_LLM = None to run the FULL 18,865-row test set (a few dollars, ~2-3 h with rate limits).
#   * Every response is cached to disk, so a re-run is free and an interrupted run resumes.

# ---- load notebooks/.env ----
try:
    from dotenv import load_dotenv
    ENV_PATH = os.environ.get("ENV_PATH", os.path.join(os.path.dirname(os.path.abspath("__file__")), ".env"))
    # also try the conventional notebooks/.env relative to ROOT, in case the notebook runs from elsewhere
    for candidate in (ENV_PATH, f"{ROOT}/notebooks/.env", "notebooks/.env", ".env"):
        if os.path.exists(candidate):
            load_dotenv(candidate, override=False)
            print(f"loaded env from: {candidate}")
            break
except ImportError:
    print("python-dotenv not installed -- run: pip install python-dotenv")

LLM_N       = None                   # None = full test set
LLM_SEED    = 42
RATE_SLEEP  = 0.4                         # seconds between calls; raise if you hit rate limits
MAX_RETRY   = 4
TEST_SPLIT  = os.environ.get("TEST_SPLIT", f"{ROOT}/data/splits/random_test.csv")
CACHE_PATH  = f"{ANN_DIR}/llm_cache.jsonl"

# ---- OpenRouter config, read from notebooks/.env ----
OPENROUTER_API_KEY  = os.environ.get("OPENROUTER_API_KEY", "")
OPENROUTER_MODEL    = os.environ.get("OPENROUTER_MODEL", "google/gemini-2.5-flash")
OPENROUTER_BASE_URL = os.environ.get("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
# NORMALIZER_PROVIDER is part of your broader project .env but is not used by this notebook.

print("OpenRouter config:")
print(f"  base_url = {OPENROUTER_BASE_URL}")
print(f"  model    = {OPENROUTER_MODEL}")
print(f"  key set  = {bool(OPENROUTER_API_KEY)}")
if not OPENROUTER_API_KEY:
    print("WARNING: OPENROUTER_API_KEY not found -- check notebooks/.env or set it in the environment.")
if "gemini" not in OPENROUTER_MODEL.lower():
    print(f"NOTE: OPENROUTER_MODEL is '{OPENROUTER_MODEL}', not a Gemini id. "
          "If you specifically want the Gemini 2.5 Flash baseline reported in the paper, "
          "set OPENROUTER_MODEL=google/gemini-2.5-flash in notebooks/.env before running.")


loaded env from: /Users/sefayet/Desktop/Github/BanglaCyberBench/notebooks/.env
OpenRouter config:
  base_url = https://openrouter.ai/api/v1
  model    = google/gemini-2.5-flash
  key set  = True


In [4]:
k = os.environ.get("OPENROUTER_API_KEY","")
print("length:", len(k))
print("prefix:", k[:12])
print("has whitespace/newline:", k != k.strip() or "\n" in k)

length: 73
prefix: sk-or-v1-976
has whitespace/newline: False


In [5]:
# ── Run the LLM zero-shot classifier via OpenRouter (cached, resumable) ──
import json, time, re

def build_prompt(comment):
    return (
        "You are labelling a single Bengali social-media comment for cyberbullying. "
        "The comment may be in Bangla script or in Romanized Bangla (Latin letters). "
        "Assign exactly ONE label from this set:\n"
        "  none      = no abuse, harassment, threat, or targeted hostility\n"
        "  abusive   = general offensive/abusive language, insults, trolling, personal attacks, slurs (not specifically sexual/religious/threat)\n"
        "  sexual    = sexual or gender-based (misogynistic) harassment\n"
        "  religious = religion-directed abuse or hostility\n"
        "  threat    = explicit threat or call to violence\n"
        "If several fit, choose by priority: threat > sexual > religious > abusive > none.\n"
        "Respond with ONLY the label word, lowercase, nothing else.\n\n"
        f"Comment: {comment}\n"
        "Label:"
    )

LABELS = ["none", "abusive", "sexual", "religious", "threat"]
def parse_label(text):
    t = str(text).strip().lower()
    for lab in LABELS:                       # exact or contained
        if re.search(rf"\b{lab}\b", t):
            return lab
    return "none"                            # safe fallback for unparseable output

def load_cache(path):
    d = {}
    if os.path.exists(path):
        for line in open(path, encoding="utf-8"):
            try:
                r = json.loads(line); d[r["uid"]] = r["pred"]
            except Exception:
                pass
    return d

def run_llm(df, text_col, uid_col):
    from openai import OpenAI    # OpenRouter is OpenAI-API-compatible
    client = OpenAI(
        api_key=OPENROUTER_API_KEY,
        base_url=OPENROUTER_BASE_URL,
        default_headers={
            "HTTP-Referer": "https://github.com/Sefayet-Alam/BanglaCyberBench",
            "X-Title": "BanglaCyberBench",
        },
    )
    cache = load_cache(CACHE_PATH)
    fout = open(CACHE_PATH, "a", encoding="utf-8")
    todo = [r for _, r in df.iterrows() if int(r[uid_col]) not in cache]
    print(f"{len(cache)} cached, {len(todo)} to call via {OPENROUTER_MODEL}")
    for i, r in enumerate(todo, 1):
        uid = int(r[uid_col]); comment = str(r[text_col])[:2000]
        pred = None
        for attempt in range(MAX_RETRY):
            try:
                resp = client.chat.completions.create(
                    model=OPENROUTER_MODEL,
                    messages=[{"role": "user", "content": build_prompt(comment)}],
                    temperature=0.0,
                    max_tokens=8,
                    extra_headers={
                        "HTTP-Referer": "https://github.com/Sefayet-Alam/BanglaCyberBench",
                        "X-Title": "BanglaCyberBench",
                    },
                )
                pred = parse_label(resp.choices[0].message.content); break
            except Exception as e:
                wait = RATE_SLEEP * (2 ** attempt)
                if attempt == MAX_RETRY - 1:
                    print(f"  uid {uid} failed ({e}); defaulting to none"); pred = "none"
                else:
                    time.sleep(wait)
        cache[uid] = pred
        fout.write(json.dumps({"uid": uid, "pred": pred}) + "\n"); fout.flush()
        time.sleep(RATE_SLEEP)
        if i % 100 == 0: print(f"  {i}/{len(todo)}")
    fout.close()
    return {int(r[uid_col]): cache.get(int(r[uid_col]), "none") for _, r in df.iterrows()}

RAN_LLM = False
if OPENROUTER_API_KEY:
    test = pd.read_csv(TEST_SPLIT, encoding="utf-8-sig").reset_index(drop=True)
    tcol = "text_clean" if "text_clean" in test.columns else "text"
    if "sample_uid" not in test.columns:
        test["sample_uid"] = test.index
    if LLM_N is not None and LLM_N < len(test):
        parts = []
        for _, grp in test.groupby("label5", sort=False):
            k = max(1, round(LLM_N * len(grp) / len(test)))
            parts.append(grp.sample(min(k, len(grp)), random_state=LLM_SEED))
        test = pd.concat(parts).reset_index(drop=True)
    print(f"LLM zero-shot ({OPENROUTER_MODEL}) over {len(test)} test comments ({tcol})")
    preds = run_llm(test, tcol, "sample_uid")
    test["llm_pred"] = test["sample_uid"].map(preds)
    RAN_LLM = True
else:
    print("skipped -- OPENROUTER_API_KEY not found. Check notebooks/.env and re-run this cell.")


LLM zero-shot (google/gemini-2.5-flash) over 18865 test comments (text_clean)
0 cached, 18865 to call via google/gemini-2.5-flash
  100/18865
  200/18865
  300/18865
  400/18865
  500/18865
  600/18865
  700/18865
  800/18865
  900/18865
  1000/18865
  1100/18865
  1200/18865
  1300/18865
  1400/18865
  1500/18865
  1600/18865
  1700/18865
  1800/18865
  1900/18865
  2000/18865
  2100/18865
  2200/18865
  2300/18865
  2400/18865
  2500/18865
  2600/18865
  2700/18865
  2800/18865
  2900/18865
  3000/18865
  3100/18865
  3200/18865
  3300/18865
  3400/18865
  3500/18865
  3600/18865
  3700/18865
  3800/18865
  3900/18865
  4000/18865
  4100/18865
  4200/18865
  4300/18865
  4400/18865
  4500/18865
  4600/18865
  4700/18865
  4800/18865
  4900/18865
  5000/18865
  5100/18865
  5200/18865
  5300/18865
  5400/18865
  5500/18865
  5600/18865
  5700/18865
  5800/18865
  5900/18865
  6000/18865
  6100/18865
  6200/18865
  6300/18865
  6400/18865
  6500/18865
  6600/18865
  6700/18865
  6800/1

In [8]:
# ── Score the Gemini baseline and write the paper table ──
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef

if RAN_LLM:
    y_true = test["label5"].map(str.lower)
    y_pred = test["llm_pred"].map(str.lower)
    m = {
        "Macro-F1":    round(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0), 4),
        "Weighted-F1": round(f1_score(y_true, y_pred, labels=LABELS, average="weighted", zero_division=0), 4),
        "Accuracy":    round(accuracy_score(y_true, y_pred), 4),
        "MCC":         round(matthews_corrcoef(y_true, y_pred), 4),
    }
    print(f"{OPENROUTER_MODEL} zero-shot:", m)

    llm_tab = pd.DataFrame([
        {"System": f"{OPENROUTER_MODEL} (zero-shot, n={len(test)})",
         "Macro-F1": f"{m['Macro-F1']:.4f}", "Weighted-F1": f"{m['Weighted-F1']:.4f}",
         "Accuracy": f"{m['Accuracy']:.4f}", "MCC": f"{m['MCC']:.4f}"},
        {"System": "Proposed model (supervised)",
         "Macro-F1": "0.8225", "Weighted-F1": "0.8332", "Accuracy": "0.8339", "MCC": "0.7452"},
    ])
    llm_tab.to_csv(f"{TAB}/table_llm_baseline.csv", index=False)
    split_desc = "full 20\\% test split" if LLM_N is None else f"stratified {len(test)}-comment test sample"
    write_tex(llm_tab, f"{TAB}/table_llm_baseline.tex",
              "Zero-shot large-language-model baseline versus the proposed supervised model on the "
              f"5-class task ({split_desc}).",
              "tab:llm", "lcccc")
    # per-class F1 for the discussion
    pcf = f1_score(y_true, y_pred, labels=LABELS, average=None, zero_division=0)
    print("per-class F1:", {l: round(float(f), 4) for l, f in zip(LABELS, pcf)})
    print(f"\n✅ wrote {TAB}/table_llm_baseline.(csv|tex)")
    print(f">>> REPORT: {OPENROUTER_MODEL} zero-shot Macro-F1 = {m['Macro-F1']:.4f} vs proposed 0.8225 <<<")
else:
    print("Part B not run — no API key. Part A (kappa) results are unaffected.")


google/gemini-2.5-flash zero-shot: {'Macro-F1': 0.5597, 'Weighted-F1': 0.6684, 'Accuracy': 0.6647, 'MCC': 0.4957}
per-class F1: {'none': 0.7765, 'abusive': 0.5733, 'sexual': 0.5413, 'religious': 0.6705, 'threat': 0.2372}

✅ wrote ../outputs/paper_q1/tables/table_llm_baseline.(csv|tex)
>>> REPORT: google/gemini-2.5-flash zero-shot Macro-F1 = 0.5597 vs proposed 0.8225 <<<
